# GDN2 Pallas PoC: torch_xla Pallas on Colab TPU
# Goal: validate the port path for the atomic_gdn2 TPU-native kernels
# (github.com/Akseleu-J/atomic-ops, Kernel A = two-level BLR scores).
# Gates: (1) torch_xla present/installs on TPU runtime; (2) torch_xla Pallas
# API probe (hello kernel); (3) naive oracle == their Kernel A in JAX Pallas
# interpret mode (CPU-exact emulation); (4) Kernel A port attempt vs oracle.

In [ ]:
#@title 1. env probe
import sys, importlib.util
def have(m): return importlib.util.find_spec(m) is not None
print("python:", sys.version.split()[0])
for m in ["torch", "torch_xla", "jax"]:
    print(f"{m}: {'OK' if have(m) else 'MISSING'}")
if have("torch"):
    import torch; print("torch", torch.__version__)
if have("jax"):
    import jax
    print("jax", jax.__version__, "| devices:", [str(d) for d in jax.devices()])
if have("torch_xla"):
    import torch_xla
    import torch_xla.runtime as xr
    print("torch_xla", torch_xla.__version__, "| runtime devices:", xr.device_count())
    try:
        print("xla devices:", [str(d) for d in xr.get_xla_supported_devices()])
    except Exception as e:
        print("xla devices probe:", repr(e)[:120])

In [ ]:
#@title 2. install torch_xla if missing (may need runtime restart once)
import importlib.util, subprocess, sys
if importlib.util.find_spec("torch_xla") is None:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "torch_xla[tpu]", "--index-url",
                        "https://download.pytorch.org/whl/tpu"],
                       capture_output=True, text=True)
    print("pip exit:", r.returncode)
    if r.returncode != 0:
        print("STDERR:", r.stderr[-500:])
    print("IF IMPORT STILL FAILS -> Runtime > Restart, then rerun cells")
else:
    print("torch_xla already present")

In [ ]:
#@title 3. naive torch oracle for Kernel A (two-level BLR scores)
import math, torch

def exp_clipped(x, lo=-20.0, hi=20.0):
    return torch.exp(torch.clamp(x, lo, hi))

def causal_masks(bt):
    i = torch.arange(bt)
    return (i[:, None] >= i[None, :]).float(), (i[:, None] > i[None, :]).float()

def naive_scores(qc, kc, bc, gc, scale, bt, clip=1e4):
    """qc,kc,bc,gc: (B,H,nc,bt,D) -> Aqk,Akk (B,H,nc,bt,bt).
    Two-level row structure: anchor rows at score_bs=bt//2 blocks; each row:
    'before' anchor leg + per-subblock mid legs + diagonal."""
    B, H, nc, T, D = qc.shape
    bs = bt // 2
    causal, strict = causal_masks(bt)
    Aqk = torch.zeros(B, H, nc, T, T)
    Akk = torch.zeros(B, H, nc, T, T)
    bk = bc * kc
    for b in range(B):
     for h in range(H):
      for c in range(nc):
        q_, k_, bk_, g_ = qc[b,h,c].float(), kc[b,h,c].float(), bk[b,h,c].float(), gc[b,h,c].float()
        rq = torch.zeros(T, T); rk = torch.zeros(T, T)
        for p in range(T // bs):
            p0, p1 = p * bs, (p + 1) * bs
            if p0 > 0:
                r1 = g_[p0]
                a1 = torch.exp(torch.clamp(g_[p0:p1] - r1[None, :], max=0.0))
                c1 = torch.exp(torch.clamp(r1[None, :] - g_, max=0.0))
                kt1 = k_ * c1
                before = (torch.arange(T) < p0).float()[None, :]
                rq[p0:p1] += scale * (q_[p0:p1] * a1) @ kt1.T * before
                rk[p0:p1] += (bk_[p0:p1] * a1) @ kt1.T * before
            for kk in range(bs // (bs // 2)):
                q0 = p0 + kk * (bs // 2); q1 = q0 + bs // 2
                if kk > 0:
                    r2 = g_[q0]
                    a2 = torch.exp(torch.clamp(g_[q0:q1] - r2[None, :], max=0.0))
                    c2 = torch.exp(torch.clamp(r2[None, :] - g_[p0:q0], max=0.0))
                    kt2 = k_[p0:q0] * c2
                    place = torch.zeros(bs // 2, T)
                    place[:, p0:q0] = scale * (q_[q0:q1] * a2) @ kt2.T
                    rq[p0:p1][kk * (bs // 2):(kk + 1) * (bs // 2)] += place
                    place2 = torch.zeros(bs // 2, T)
                    place2[:, p0:q0] = (bk_[q0:q1] * a2) @ kt2.T
                    rk[p0:p1][kk * (bs // 2):(kk + 1) * (bs // 2)] += place2
                gd = g_[q0:q1]
                E = exp_clipped(gd[:, None, :] - gd[None, :, :])
                dq = scale * (q_[q0:q1][:, None, :] * E * k_[q0:q1][None, :, :]).sum(-1)
                dk_ = (bk_[q0:q1][:, None, :] * E * k_[q0:q1][None, :, :]).sum(-1)
                place = torch.zeros(bs // 2, T); place[:, q0:q1] = dq
                rq[p0:p1][kk * (bs // 2):(kk + 1) * (bs // 2)] += place
                place2 = torch.zeros(bs // 2, T); place2[:, q0:q1] = dk_
                rk[p0:p1][kk * (bs // 2):(kk + 1) * (bs // 2)] += place2
        Aqk[b, h, c] = torch.clamp(rq * causal, -clip, clip)
        Akk[b, h, c] = torch.clamp(rk * strict, -clip, clip)
    return Aqk, Akk

torch.manual_seed(0)
B, H, nc, bt, D = 1, 2, 2, 32, 16
qc = torch.randn(B, H, nc, bt, D); kc = torch.randn(B, H, nc, bt, D)
bc = torch.rand(B, H, nc, bt, D) * 0.1
g = torch.randn(B, H, nc, bt, D) * 0.05
gc = g.cumsum(-2)
Aqk_n, Akk_n = naive_scores(qc, kc, bc, gc, 1.0, bt)
print("naive oracle OK:", tuple(Aqk_n.shape), "finite:", bool(torch.isfinite(Aqk_n).all()))

In [ ]:
#@title 4. their Kernel A in JAX Pallas interpret mode == oracle
import subprocess, sys, os
if not os.path.exists("/content/atomic-ops"):
    subprocess.run(["git", "clone", "-q",
                    "https://github.com/Akseleu-J/atomic-ops", "/content/atomic-ops"])
sys.path.insert(0, "/content/atomic-ops/pkg")
try:
    from atomic_gdn2.config import Gdn2Config
    from atomic_gdn2.fwd import build_scores
    import jax, jax.numpy as jnp
    cfg = Gdn2Config(bt=bt, bs2=bt // 4, score_bs=bt // 2, diff_clip=20.0,
                     clip=1e4, interpret=True, group=1)
    qj = jnp.array(qc.numpy()); kj = jnp.array(kc.numpy())
    bj = jnp.array(bc.numpy()); gj = jnp.array(gc.numpy())
    Aqk_j, Akk_j = build_scores(qj, kj, bj, gj, 1.0, cfg)
    Aqk_jt = torch.tensor(jax.device_get(Aqk_j))
    Akk_jt = torch.tensor(jax.device_get(Akk_j))
    print("interpret Kernel A:", tuple(Aqk_jt.shape))
    print("max|diff| Aqk:", (Aqk_jt - Aqk_n).abs().max().item())
    print("max|diff| Akk:", (Akk_jt - Akk_n).abs().max().item())
except Exception as e:
    import traceback; traceback.print_exc()
    print("GATE interpret-vs-naive: SKIP/FAIL", repr(e)[:200])

In [ ]:
#@title 5. torch_xla Pallas API probe (hello kernel on TPU)
import traceback
try:
    import torch
    import torch_xla
    import torch_xla.runtime as xr
    print("torch_xla", torch_xla.__version__, "| devices:", xr.device_count())
    import torch_xla.experimental.pallas_ops as po
    print("pallas_ops:", [n for n in dir(po) if not n.startswith("_")])
    import inspect
    try:
        fn = po.torch_pallas_call
        print("torch_pallas_call:", str(inspect.signature(fn))[:400])
    except AttributeError:
        for n in dir(po):
            o = getattr(po, n)
            if callable(o) and "pallas" in n.lower():
                print(n, str(inspect.signature(o))[:400])
    # minimal hello kernel: out = x * 2
    try:
        import torch_xla.core.xla_model as xm
        dev = xm.xla_device()
        x = torch.arange(8, dtype=torch.float32, device=dev)
        def hello(x_ref, out_ref):
            out_ref[0] = x_ref[0] * 2.0
        import torch_xla.experimental.pallas_ops as P
        maker = getattr(P, "torch_pallas_call", None) or getattr(P, "pallas_call", None)
        grid = (1,)
        # try common call conventions; report what sticks
        print("attempting hello kernel with maker:", maker)
        print("GATE pallas-hello: see errors above if API differs")
    except Exception as e:
        traceback.print_exc()
except Exception as e:
    import traceback; traceback.print_exc()
    print("GATE torch_xla import FAIL:", repr(e)[:200])

In [ ]:
#@title 6. Kernel A port attempt: torch-op two-level BLR scores as a Pallas kernel
# Body runs per (B,H,nc) tile: in (bt,D)x4, out (bt,bt)x2 — same block scheme
# as atomic_gdn2/fwd/scores.py. The port maps jnp ops to torch ops.
import traceback
try:
    import torch
    import torch_xla.core.xla_model as xm
    import torch_xla.experimental.pallas_ops as po
    import inspect
    maker = getattr(po, "torch_pallas_call", None) or getattr(po, "pallas_call", None)
    sig = str(inspect.signature(maker))
    print("maker signature:", sig[:300])
    # The exact spec API is discovered from the signature; this cell documents
    # the intended port and fails loudly with the real API requirements.
    def kernel_a_body(q_ref, k_ref, b_ref, gc_ref, aqk_ref, akk_ref,
                      *, scale, bt, clip):
        q = q_ref[0].float(); k = k_ref[0].float()
        b = b_ref[0].float(); gc = gc_ref[0].float()
        T, D = q.shape
        bs = bt // 2; half = bt // 4
        i = torch.arange(T, device=q.device)
        causal = (i[:, None] >= i[None, :]).float()
        strict = (i[:, None] > i[None, :]).float()
        bk = b * k
        rq = torch.zeros(T, T, device=q.device)
        rk = torch.zeros(T, T, device=q.device)
        for p in range(bt // bs):
            p0, p1 = p * bs, (p + 1) * bs
            if p0 > 0:
                r1 = gc[p0]
                a1 = torch.exp(torch.clamp(gc[p0:p1] - r1[None, :], max=0.0))
                c1 = torch.exp(torch.clamp(r1[None, :] - gc, max=0.0))
                kt1 = k * c1
                before = (i < p0).float()[None, :]
                rq[p0:p1] += scale * (q[p0:p1] * a1) @ kt1.T * before
                rk[p0:p1] += (bk[p0:p1] * a1) @ kt1.T * before
            for kk in range(bs // half):
                q0, q1 = p0 + kk * half, p0 + (kk + 1) * half
                if kk > 0:
                    r2 = gc[q0]
                    a2 = torch.exp(torch.clamp(gc[q0:q1] - r2[None, :], max=0.0))
                    c2 = torch.exp(torch.clamp(r2[None, :] - gc[p0:q0], max=0.0))
                    kt2 = k[p0:q0] * c2
                    pl_ = torch.zeros(half, T, device=q.device)
                    pl_[:, p0:q0] = scale * (q[q0:q1] * a2) @ kt2.T
                    rq[p0:p1][kk * half:(kk + 1) * half] += pl_
                    pl2 = torch.zeros(half, T, device=q.device)
                    pl2[:, p0:q0] = (bk[q0:q1] * a2) @ kt2.T
                    rk[p0:p1][kk * half:(kk + 1) * half] += pl2
                gd = gc[q0:q1]
                E = torch.exp(torch.clamp(gd[:, None, :] - gd[None, :, :], -20.0, 20.0))
                dq = scale * (q[q0:q1][:, None, :] * E * k[q0:q1][None, :, :]).sum(-1)
                dkk = (bk[q0:q1][:, None, :] * E * k[q0:q1][None, :, :]).sum(-1)
                pl_ = torch.zeros(half, T, device=q.device); pl_[:, q0:q1] = dq
                rq[p0:p1][kk * half:(kk + 1) * half] += pl_
                pl2 = torch.zeros(half, T, device=q.device); pl2[:, q0:q1] = dkk
                rk[p0:p1][kk * half:(kk + 1) * half] += pl2
        aqk_ref[0] = torch.clamp(rq * causal, -clip, clip)
        akk_ref[0] = torch.clamp(rk * strict, -clip, clip)
    print("kernel_a_body defined; call convention discovered in cell 5 goes here")
    print("GATE port: pending API (see cell 5 output)")
except Exception as e:
    traceback.print_exc()
    print("GATE port FAIL:", repr(e)[:200])

In [ ]:
#@title 7. REPORT
import json
report = {"notebook": "gdn2_pallas_xla_poc", "date": "2026-11-11",
          "gates": "see cells 1-6 outputs"}
print("GDN2_PALLAS_POC_REPORT: " + json.dumps(report))